# Network-constrained electricity market clearing

This notebook implements the simplified **social-surplus-maximizing linear program** for the 13 European bidding-area files in `data/`.

For a selected 15-minute delivery period, each cumulative supply and demand curve is converted into incremental bid blocks. The optimizer jointly chooses accepted supply, accepted demand, and inter-area power flows:

\[
\max \quad \sum_{i,m} v_{im} d_{im} - \sum_{i,k} c_{ik} s_{ik}
\]

subject to nodal balance

\[
\sum_k s_{ik} + \sum_j f_{ji} = \sum_m d_{im} + \sum_j f_{ij},
\]

bid-block bounds

\[
0 \le s_{ik} \le \Delta Q^S_{ik}, \qquad
0 \le d_{im} \le \Delta Q^D_{im},
\]

and symmetric interconnector limits

\[
-F_{ij} \le f_{ij} \le F_{ij}.
\]

The notebook compares autarky with an editable, capacity-constrained network. It deliberately does **not** extrapolate beyond submitted bids and does not yet model AC power flow, losses, unit commitment, or investment.

## 1. Imports and user configuration

Change `PERIOD_INDEX` to select one of the 96 quarter-hours. The first period is index `0`; the last is index `95`.

The capacity table later in the notebook is an **illustrative, editable baseline**, not a claim about historical net transfer capacities. Replace those values with the capacity dataset used in the final analysis. Its default scale is intentionally conservative so that the example produces visible congestion diagnostics.

In [ ]:
from pathlib import Path
import json
import math

import gurobipy as gp
from gurobipy import GRB
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

# The notebook is designed to be run from the D2A_Project_Draft_1 folder.
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
PERIOD_INDEX = 0
PLOT_MARKET = "AT"

if not DATA_DIR.exists():
    raise FileNotFoundError(
        "Could not find ./data. Start Jupyter in D2A_Project_Draft_1, "
        "or set PROJECT_DIR above to that folder."
    )

print(f"Project directory: {PROJECT_DIR}")
print(f"Gurobi version: {gp.gurobi.version()}")

## 2. Load and validate the market files

In [ ]:
def load_market_files(data_dir):
    """Load one JSON-formatted .txt file per aggregate bidding area."""
    markets = {}
    source_paths = {}

    for path in sorted(Path(data_dir).glob("*.txt")):
        with path.open("r", encoding="utf-8") as handle:
            payload = json.load(handle)

        area = payload.get("aggregatedBiddingArea")
        periods = payload.get("aggregatedPoints", [])
        if not area or not periods:
            raise ValueError(f"{path.name} is missing its area code or delivery periods")
        if area in markets:
            raise ValueError(f"Duplicate aggregate bidding area: {area}")

        markets[area] = payload
        source_paths[area] = path

    if not markets:
        raise FileNotFoundError(f"No .txt market files found in {data_dir}")

    period_counts = {area: len(payload["aggregatedPoints"]) for area, payload in markets.items()}
    if len(set(period_counts.values())) != 1:
        raise ValueError(f"Market files have inconsistent period counts: {period_counts}")

    return markets, source_paths


markets, source_paths = load_market_files(DATA_DIR)

market_overview = pd.DataFrame(
    [
        {
            "market": area,
            "periods": len(payload["aggregatedPoints"]),
            "delivery_date": payload.get("deliveryDate"),
            "file": source_paths[area].name,
        }
        for area, payload in sorted(markets.items())
    ]
)

print(f"Loaded {len(markets)} markets.")
display(market_overview)

## 3. Convert cumulative schedules into incremental blocks

The raw curves are cumulative. Whenever cumulative quantity rises, the increase becomes one bounded bid block. A vertical price move with no quantity change creates no block. Adjacent blocks at the same price are merged.

The first cumulative quantity is treated as a block measured from zero. This preserves the absolute submitted supply and demand schedules rather than re-centering each market around its observed clearing point.

In [ ]:
def cumulative_curve_to_blocks(series_data, side, quantity_tolerance=1e-7, price_tolerance=1e-10):
    """Convert one cumulative supply or demand schedule to (price, quantity) blocks."""
    if side not in {"supply", "demand"}:
        raise ValueError("side must be 'supply' or 'demand'")

    price_key = f"{side}Price"
    quantity_key = f"{side}Quantity"
    points = [
        (float(row[price_key]), float(row[quantity_key]))
        for row in series_data
        if price_key in row and quantity_key in row
    ]
    if not points:
        raise ValueError(f"No complete {side} price-quantity points found")

    blocks = []
    previous_quantity = 0.0
    for price, cumulative_quantity in points:
        if not (math.isfinite(price) and math.isfinite(cumulative_quantity)):
            raise ValueError(f"Non-finite {side} point: {(price, cumulative_quantity)}")
        if cumulative_quantity < previous_quantity - quantity_tolerance:
            raise ValueError(
                f"{side.title()} cumulative quantity fell from "
                f"{previous_quantity} to {cumulative_quantity}"
            )

        block_quantity = cumulative_quantity - previous_quantity
        if block_quantity > quantity_tolerance:
            if blocks and abs(blocks[-1]["price"] - price) <= price_tolerance:
                blocks[-1]["quantity"] += block_quantity
            else:
                blocks.append({"price": price, "quantity": block_quantity})
        previous_quantity = max(previous_quantity, cumulative_quantity)

    result = pd.DataFrame(blocks)
    result.insert(0, "block_id", np.arange(len(result), dtype=int))
    result["side"] = side
    return result[["block_id", "side", "price", "quantity"]]


def published_price_summary(period):
    """Return published component prices and their mean for an aggregate file."""
    records = period.get("priceAreas", [])
    prices = {
        str(record["area"]): float(record["price"])
        for record in records
        if "area" in record and "price" in record
    }
    mean_price = float(np.mean(list(prices.values()))) if prices else np.nan
    return prices, mean_price


def prepare_period(markets, period_index):
    """Parse the selected period for every market into optimizer-ready blocks."""
    prepared = {}
    delivery_starts = set()
    delivery_ends = set()

    for area, payload in sorted(markets.items()):
        periods = payload["aggregatedPoints"]
        if period_index < 0 or period_index >= len(periods):
            raise IndexError(f"PERIOD_INDEX={period_index} is invalid for {area}")

        period = periods[period_index]
        delivery_starts.add(period["deliveryStart"])
        delivery_ends.add(period["deliveryEnd"])
        prices, mean_price = published_price_summary(period)
        prepared[area] = {
            "supply": cumulative_curve_to_blocks(period["seriesData"], "supply"),
            "demand": cumulative_curve_to_blocks(period["seriesData"], "demand"),
            "published_prices": prices,
            "published_price_mean": mean_price,
            "delivery_start": period["deliveryStart"],
            "delivery_end": period["deliveryEnd"],
        }

    if len(delivery_starts) != 1 or len(delivery_ends) != 1:
        raise ValueError(
            "The selected period is not time-aligned across markets: "
            f"starts={delivery_starts}, ends={delivery_ends}"
        )

    return prepared


period_data = prepare_period(markets, PERIOD_INDEX)
delivery_start = next(iter(period_data.values()))["delivery_start"]
delivery_end = next(iter(period_data.values()))["delivery_end"]

block_summary = pd.DataFrame(
    [
        {
            "market": area,
            "supply_blocks": len(data["supply"]),
            "max_supply_mw": data["supply"]["quantity"].sum(),
            "demand_blocks": len(data["demand"]),
            "max_demand_mw": data["demand"]["quantity"].sum(),
            "published_price_mean": data["published_price_mean"],
        }
        for area, data in sorted(period_data.items())
    ]
)

print(f"Selected period {PERIOD_INDEX}: {delivery_start} to {delivery_end}")
display(block_summary)

### Inspect one transformed market

In [ ]:
if PLOT_MARKET not in period_data:
    raise KeyError(f"PLOT_MARKET={PLOT_MARKET!r} is not one of {sorted(period_data)}")

supply_plot = period_data[PLOT_MARKET]["supply"].copy()
demand_plot = period_data[PLOT_MARKET]["demand"].copy()
supply_plot["cumulative_quantity"] = supply_plot["quantity"].cumsum()
demand_plot["cumulative_quantity"] = demand_plot["quantity"].cumsum()

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.step(
    supply_plot["cumulative_quantity"], supply_plot["price"],
    where="post", label="Supply", linewidth=2,
)
ax.step(
    demand_plot["cumulative_quantity"], demand_plot["price"],
    where="post", label="Demand", linewidth=2,
)
ax.set_title(f"{PLOT_MARKET}: submitted bid schedules, period {PERIOD_INDEX}")
ax.set_xlabel("Cumulative quantity (MW)")
ax.set_ylabel("Price (EUR/MWh)")
ax.grid(alpha=0.25)
ax.legend()
plt.show()

display(supply_plot.head(10).style.set_caption("First 10 incremental supply blocks"))
display(demand_plot.head(10).style.set_caption("First 10 incremental demand blocks"))

## 4. Define the transport network

Each row is one undirected commercial interconnector. The flow variable is signed: positive flow goes from `from_market` to `to_market`, and negative flow goes in the reverse direction.

**Important:** the numbers below are round, illustrative capacities included so the notebook runs end-to-end. They are not a verified ENTSO-E capacity dataset. The default `ILLUSTRATIVE_CAPACITY_SCALE` is `0.5` so that the demonstration contains economically scarce links. Edit this cell before treating the quantitative results as empirical findings.

In [ ]:
BASE_INTERCONNECTORS = pd.DataFrame(
    [
        # Central-western Europe
        ("AT", "DE-LU", 4_000.0),
        ("BE", "DE-LU", 2_000.0),
        ("BE", "FR", 3_500.0),
        ("BE", "NL", 2_400.0),
        ("DE-LU", "FR", 4_000.0),
        ("DE-LU", "NL", 5_000.0),
        ("DE-LU", "PL", 2_500.0),
        # Nordic and continental links
        ("DE-LU", "DK", 2_500.0),
        ("DE-LU", "SE", 600.0),
        ("DK", "NL", 700.0),
        ("DK", "NO", 1_700.0),
        ("DK", "SE", 2_000.0),
        ("FI", "NO", 100.0),
        ("FI", "SE", 2_700.0),
        ("NO", "NL", 700.0),
        ("NO", "SE", 3_500.0),
        # Baltic and eastern links
        ("BALTIC", "FI", 1_000.0),
        ("BALTIC", "PL", 500.0),
        ("BALTIC", "SE", 700.0),
        ("PL", "SE", 600.0),
        ("BG", "RO", 1_500.0),
    ],
    columns=["from_market", "to_market", "capacity_mw"],
)

# A conservative demonstration scale that makes transfer scarcity visible.
# Set this to 1.0 to use the round base values exactly as listed above.
ILLUSTRATIVE_CAPACITY_SCALE = 0.5
INTERCONNECTORS = BASE_INTERCONNECTORS.copy()
INTERCONNECTORS["capacity_mw"] *= ILLUSTRATIVE_CAPACITY_SCALE


def validate_interconnectors(interconnectors, valid_markets):
    required = {"from_market", "to_market", "capacity_mw"}
    if not required.issubset(interconnectors.columns):
        raise ValueError(f"Interconnector table must contain {sorted(required)}")

    unknown = (
        set(interconnectors["from_market"])
        | set(interconnectors["to_market"])
    ) - set(valid_markets)
    if unknown:
        raise ValueError(f"Unknown markets in interconnector table: {sorted(unknown)}")
    if (interconnectors["from_market"] == interconnectors["to_market"]).any():
        raise ValueError("Self-links are not allowed")
    if (interconnectors["capacity_mw"] < 0).any():
        raise ValueError("Capacities must be nonnegative")

    normalized_pairs = interconnectors.apply(
        lambda row: tuple(sorted((row["from_market"], row["to_market"]))), axis=1
    )
    if normalized_pairs.duplicated().any():
        duplicates = normalized_pairs[normalized_pairs.duplicated(keep=False)].tolist()
        raise ValueError(f"Duplicate undirected links: {duplicates}")


validate_interconnectors(INTERCONNECTORS, period_data)
display(INTERCONNECTORS)

## 5. Build and solve the Gurobi linear program

In [ ]:
def solve_market(period_data, interconnectors, model_name="market_clearing", output_flag=0):
    """Solve one-period surplus-maximizing market clearing and return tidy results."""
    nodes = sorted(period_data)
    edges = interconnectors.reset_index(drop=True).copy()
    validate_interconnectors(edges, nodes)

    model = gp.Model(model_name)
    model.Params.OutputFlag = output_flag
    model.Params.Method = 1  # dual simplex gives stable LP dual/reduced-cost diagnostics

    supply_vars = {}
    demand_vars = {}
    flow_vars = {}

    for node in nodes:
        for row in period_data[node]["supply"].itertuples(index=False):
            supply_vars[(node, row.block_id)] = model.addVar(
                lb=0.0,
                ub=float(row.quantity),
                obj=-float(row.price),
                name=f"supply[{node},{row.block_id}]",
            )
        for row in period_data[node]["demand"].itertuples(index=False):
            demand_vars[(node, row.block_id)] = model.addVar(
                lb=0.0,
                ub=float(row.quantity),
                obj=float(row.price),
                name=f"demand[{node},{row.block_id}]",
            )

    for edge_id, row in edges.iterrows():
        capacity = float(row["capacity_mw"])
        flow_vars[edge_id] = model.addVar(
            lb=-capacity,
            ub=capacity,
            obj=0.0,
            name=f"flow[{row['from_market']}->{row['to_market']}]",
        )

    model.ModelSense = GRB.MAXIMIZE
    model.update()

    balance_constraints = {}
    for node in nodes:
        expression = gp.quicksum(
            supply_vars[(node, block_id)]
            for block_id in period_data[node]["supply"]["block_id"]
        )
        expression -= gp.quicksum(
            demand_vars[(node, block_id)]
            for block_id in period_data[node]["demand"]["block_id"]
        )

        for edge_id, edge in edges.iterrows():
            if node == edge["from_market"]:
                expression -= flow_vars[edge_id]
            elif node == edge["to_market"]:
                expression += flow_vars[edge_id]

        balance_constraints[node] = model.addConstr(
            expression == 0.0,
            name=f"balance[{node}]",
        )

    model.optimize()
    if model.Status != GRB.OPTIMAL:
        raise RuntimeError(f"Gurobi did not find an optimum; status={model.Status}")

    node_rows = []
    total_supply_cost = 0.0
    total_demand_benefit = 0.0

    for node in nodes:
        supply_blocks = period_data[node]["supply"].set_index("block_id")
        demand_blocks = period_data[node]["demand"].set_index("block_id")
        accepted_supply = sum(
            variable.X for (area, _), variable in supply_vars.items() if area == node
        )
        accepted_demand = sum(
            variable.X for (area, _), variable in demand_vars.items() if area == node
        )
        supply_cost = sum(
            float(supply_blocks.loc[block_id, "price"]) * variable.X
            for (area, block_id), variable in supply_vars.items()
            if area == node
        )
        demand_benefit = sum(
            float(demand_blocks.loc[block_id, "price"]) * variable.X
            for (area, block_id), variable in demand_vars.items()
            if area == node
        )
        total_supply_cost += supply_cost
        total_demand_benefit += demand_benefit

        # With balance written as supply + imports - demand - exports = 0,
        # the economically signed nodal price is the negative Gurobi dual.
        nodal_price = -float(balance_constraints[node].Pi)
        node_rows.append(
            {
                "market": node,
                "supply_mw": accepted_supply,
                "demand_mw": accepted_demand,
                "net_export_mw": accepted_supply - accepted_demand,
                "nodal_price_eur_per_mwh": nodal_price,
                "published_price_mean": period_data[node]["published_price_mean"],
                "price_error": nodal_price - period_data[node]["published_price_mean"],
                "gross_benefit_eur_per_h": demand_benefit,
                "supply_cost_eur_per_h": supply_cost,
                "balance_residual_mw": float(balance_constraints[node].Slack),
            }
        )

    node_results = pd.DataFrame(node_rows).sort_values("market").reset_index(drop=True)
    node_prices = node_results.set_index("market")["nodal_price_eur_per_mwh"].to_dict()

    flow_rows = []
    for edge_id, edge in edges.iterrows():
        variable = flow_vars[edge_id]
        signed_flow = float(variable.X)
        capacity = float(edge["capacity_mw"])
        utilization = abs(signed_flow) / capacity if capacity > 0 else np.nan
        at_limit = capacity > 0 and abs(abs(signed_flow) - capacity) <= max(1e-5, 1e-6 * capacity)
        marginal_capacity_value = abs(float(variable.RC)) if at_limit else 0.0
        economically_binding = at_limit and marginal_capacity_value > 1e-6

        if signed_flow >= 0:
            actual_from, actual_to = edge["from_market"], edge["to_market"]
        else:
            actual_from, actual_to = edge["to_market"], edge["from_market"]

        price_difference = node_prices[actual_to] - node_prices[actual_from]
        flow_rows.append(
            {
                "configured_link": f"{edge['from_market']} -- {edge['to_market']}",
                "actual_direction": f"{actual_from} -> {actual_to}",
                "signed_flow_mw": signed_flow,
                "flow_mw": abs(signed_flow),
                "capacity_mw": capacity,
                "utilization": utilization,
                "at_limit": at_limit,
                "economically_binding": economically_binding,
                "receiving_minus_sending_price": price_difference,
                "marginal_value_of_more_capacity": marginal_capacity_value,
                "congestion_rent_eur_per_h": abs(signed_flow) * abs(price_difference),
            }
        )

    flow_results = pd.DataFrame(flow_rows)
    if not flow_results.empty:
        flow_results = flow_results.sort_values(
            ["economically_binding", "at_limit", "utilization"],
            ascending=[False, False, False],
        ).reset_index(drop=True)

    objective_check = total_demand_benefit - total_supply_cost
    if not np.isclose(model.ObjVal, objective_check, rtol=1e-8, atol=1e-4):
        raise AssertionError(
            f"Objective mismatch: model={model.ObjVal}, reconstructed={objective_check}"
        )

    return {
        "model": model,
        "nodes": node_results,
        "flows": flow_results,
        "welfare": float(model.ObjVal),
        "gross_benefit": total_demand_benefit,
        "supply_cost": total_supply_cost,
        "delivery_start": next(iter(period_data.values()))["delivery_start"],
        "delivery_end": next(iter(period_data.values()))["delivery_end"],
    }

## 6. Compare autarky and constrained trade

Autarky uses the same bid blocks but no inter-area links. Therefore, the welfare difference isolates the value of trade permitted by the configured transport network for this one 15-minute market-clearing interval.

The objective is reported in EUR/hour because prices are EUR/MWh and quantities are MW. Multiply the objective or welfare gain by `0.25` to obtain EUR for a 15-minute interval.

In [ ]:
AUTARKY_LINKS = pd.DataFrame(
    columns=["from_market", "to_market", "capacity_mw"]
)

autarky = solve_market(period_data, AUTARKY_LINKS, model_name="autarky")
network = solve_market(period_data, INTERCONNECTORS, model_name="constrained_network")

scenario_summary = pd.DataFrame(
    [
        {
            "scenario": "Autarky",
            "welfare_eur_per_h": autarky["welfare"],
            "welfare_for_15min_eur": 0.25 * autarky["welfare"],
            "gain_vs_autarky_eur_per_h": 0.0,
            "gain_vs_autarky_for_15min_eur": 0.0,
        },
        {
            "scenario": "Capacity-constrained network",
            "welfare_eur_per_h": network["welfare"],
            "welfare_for_15min_eur": 0.25 * network["welfare"],
            "gain_vs_autarky_eur_per_h": network["welfare"] - autarky["welfare"],
            "gain_vs_autarky_for_15min_eur": 0.25 * (network["welfare"] - autarky["welfare"]),
        },
    ]
)

display(scenario_summary)

### Nodal quantities and prices

In [ ]:
node_comparison = (
    autarky["nodes"][[
        "market", "supply_mw", "demand_mw", "nodal_price_eur_per_mwh"
    ]]
    .rename(
        columns={
            "supply_mw": "autarky_supply_mw",
            "demand_mw": "autarky_demand_mw",
            "nodal_price_eur_per_mwh": "autarky_price",
        }
    )
    .merge(
        network["nodes"][[
            "market", "supply_mw", "demand_mw", "net_export_mw",
            "nodal_price_eur_per_mwh", "published_price_mean", "price_error"
        ]],
        on="market",
        how="inner",
    )
    .rename(
        columns={
            "supply_mw": "network_supply_mw",
            "demand_mw": "network_demand_mw",
            "nodal_price_eur_per_mwh": "network_price",
        }
    )
)

display(node_comparison)

### Interconnector use and congestion diagnostics

In [ ]:
flow_display = network["flows"].copy()
flow_display["utilization_pct"] = 100.0 * flow_display["utilization"]
display(
    flow_display[[
        "configured_link", "actual_direction", "flow_mw", "capacity_mw",
        "utilization_pct", "at_limit", "economically_binding",
        "receiving_minus_sending_price",
        "marginal_value_of_more_capacity", "congestion_rent_eur_per_h",
    ]]
)

at_limit_links = flow_display.loc[flow_display["at_limit"]]
scarce_links = flow_display.loc[flow_display["economically_binding"]]
print(f"Interconnectors at a limit: {len(at_limit_links)} of {len(flow_display)}")
print(f"Economically binding interconnectors: {len(scarce_links)} of {len(flow_display)}")

### Visual summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

price_plot = node_comparison.sort_values("network_price")
x = np.arange(len(price_plot))
width = 0.38
axes[0].bar(x - width / 2, price_plot["autarky_price"], width, label="Autarky")
axes[0].bar(x + width / 2, price_plot["network_price"], width, label="Network")
axes[0].set_xticks(x, price_plot["market"], rotation=45, ha="right")
axes[0].set_ylabel("Nodal price (EUR/MWh)")
axes[0].set_title("Price convergence from constrained trade")
axes[0].legend()
axes[0].grid(axis="y", alpha=0.25)

utilization_plot = flow_display.sort_values("utilization_pct", ascending=True)
colors = np.where(utilization_plot["economically_binding"], "tab:red", "tab:blue")
axes[1].barh(
    utilization_plot["configured_link"],
    utilization_plot["utilization_pct"],
    color=colors,
)
axes[1].axvline(100, color="black", linewidth=1, linestyle="--")
axes[1].set_xlabel("Absolute utilization (%)")
axes[1].set_title("Interconnector utilization (red = economically binding)")
axes[1].grid(axis="x", alpha=0.25)

fig.tight_layout()
plt.show()

## 7. Numerical checks and interpretation

- `balance_residual_mw` should be essentially zero at every node.
- `at_limit` means the returned flow equals a capacity bound. In a transport model with costless routing, some at-limit flows can be part of a degenerate circulation and have zero economic value.
- `economically_binding` additionally requires a positive reduced-cost scarcity value.
- `marginal_value_of_more_capacity` is the local welfare gain from relaxing an economically binding symmetric flow bound by approximately 1 MW, holding the rest of this one-period model fixed.
- Price differences across uncongested connected areas should be near zero. Persistent differences across a used link indicate scarcity of transfer capacity.
- Published-price comparisons are diagnostic only. The simplified model omits losses, loop flows, separate sub-areas, reserves, nonconvex bids, unit commitment, and other market rules.

In [ ]:
diagnostics = pd.Series(
    {
        "max_abs_balance_residual_mw": network["nodes"]["balance_residual_mw"].abs().max(),
        "system_net_export_mw": network["nodes"]["net_export_mw"].sum(),
        "interconnectors_at_limit": int(network["flows"]["at_limit"].sum()),
        "economically_binding_interconnectors": int(
            network["flows"]["economically_binding"].sum()
        ),
        "welfare_gain_eur_per_h": network["welfare"] - autarky["welfare"],
        "welfare_gain_for_15min_eur": 0.25 * (network["welfare"] - autarky["welfare"]),
    },
    name="value",
)
display(diagnostics.to_frame())

assert diagnostics["max_abs_balance_residual_mw"] <= 1e-5
assert abs(diagnostics["system_net_export_mw"]) <= 1e-5
assert network["welfare"] >= autarky["welfare"] - 1e-4
print("All numerical checks passed.")

## 8. Optional multi-period sensitivity

The same model can be run on several of the 96 delivery periods without extrapolating any curve. Set the switch below to `True` to compare representative overnight, morning, afternoon, and evening snapshots.

For a full-day welfare total, solve all 96 periods and multiply each EUR/hour objective by 0.25 hours before summing.

In [ ]:
RUN_MULTI_PERIOD_SENSITIVITY = False
REPRESENTATIVE_PERIODS = [0, 28, 52, 76]


def run_period_sensitivity(markets, period_indices, interconnectors):
    rows = []
    for period_index in period_indices:
        one_period = prepare_period(markets, period_index)
        autarky_result = solve_market(
            one_period,
            AUTARKY_LINKS,
            model_name=f"autarky_period_{period_index}",
        )
        network_result = solve_market(
            one_period,
            interconnectors,
            model_name=f"network_period_{period_index}",
        )
        rows.append(
            {
                "period_index": period_index,
                "delivery_start": network_result["delivery_start"],
                "delivery_end": network_result["delivery_end"],
                "autarky_welfare_eur_per_h": autarky_result["welfare"],
                "network_welfare_eur_per_h": network_result["welfare"],
                "gain_for_15min_eur": 0.25 * (
                    network_result["welfare"] - autarky_result["welfare"]
                ),
                "economically_binding_links": int(
                    network_result["flows"]["economically_binding"].sum()
                ),
            }
        )
    return pd.DataFrame(rows)


if RUN_MULTI_PERIOD_SENSITIVITY:
    sensitivity_results = run_period_sensitivity(
        markets,
        REPRESENTATIVE_PERIODS,
        INTERCONNECTORS,
    )
    display(sensitivity_results)

## Suggested next extensions

1. Replace the illustrative capacities with sourced, date-consistent interconnector data.
2. Model the component bidding zones separately where an aggregate file combines areas with different published prices.
3. Solve all 96 periods and summarize total daily welfare and recurring bottlenecks.
4. Add capacity-expansion variables and a budget only after the fixed-network baseline is validated.
5. If the research question requires physical power flows, replace the transport representation with a DC power-flow approximation before introducing resistive losses.